# Practo Doctor Scraper

This notebook scrapes Practo doctor listings for 10 major cities and collects 50 doctors per city, for a total of 500 doctors.

Target fields:
- doctor_name
- doctor_type
- experience_years
- location
- hospital
- fees
- city

The code uses the Practo search page and extracts values from the visible listing cards and the embedded JSON-LD metadata when available.

In [1]:
import json
import re
import time
from urllib.parse import quote

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 '
        '(KHTML, like Gecko) Chrome/125.0 Safari/537.36'
    ),
    'Accept-Language': 'en-US,en;q=0.9',
}

CITIES = [
    'Bangalore',
    'Delhi',
    'Mumbai',
    'Hyderabad',
    'Chennai',
    'Kolkata',
    'Pune',
    'Ahmedabad',
    'Jaipur',
    'Chandigarh',
]

BASE_URL = 'https://www.practo.com/search/doctors'
SEARCH_Q = '%5B%7B%22word%22%3A%22doctor%22%2C%22autocompleted%22%3Atrue%2C%22category%22%3A%22common_name%22%7D%5D'

def build_url(city, page=1):
    return f'{BASE_URL}?results_type=doctor&q={SEARCH_Q}&city={quote(city)}&page={page}'

In [8]:
def parse_fee_from_text(text):
    match = re.search(r'₹\s*([\d,]+)', text)
    if match:
        return int(match.group(1).replace(',', ''))
    return None


def parse_experience(text):
    match = re.search(r'(\d+)\s+years? experience overall', text, flags=re.I)
    if match:
        return int(match.group(1))
    return None


def parse_card(card, city):
    name_tag = card.select_one("h2[data-qa-id='doctor_name']")
    specialty_tag = card.select_one('div.u-grey_3-text span')
    exp_tag = card.select_one("div[data-qa-id='doctor_experience']")
    locality_tag = card.select_one("span[data-qa-id='practice_locality']")
    city_tag = card.select_one("span[data-qa-id='practice_city']")
    ld_json = card.find('script', type='application/ld+json')
    hospital_link = card.find('a', href=re.compile(r'/(hospital|clinic)/', re.I))

    meta = {}
    if ld_json and ld_json.string:
        try:
            meta = json.loads(ld_json.string)
        except json.JSONDecodeError:
            meta = {}

    raw_text = card.get_text(' ', strip=True)
    doctor_name = name_tag.get_text(' ', strip=True) if name_tag else None
    doctor_type = specialty_tag.get_text(' ', strip=True) if specialty_tag else None
    experience_years = parse_experience(exp_tag.get_text(' ', strip=True)) if exp_tag else None
    location_parts = []
    if locality_tag:
        location_parts.append(locality_tag.get_text(' ', strip=True).replace(',', ''))
    if city_tag:
        location_parts.append(city_tag.get_text(' ', strip=True))
    location = ', '.join([part for part in location_parts if part]) or city
    hospital = meta.get('branchOf', {}).get('name')
    if not hospital and hospital_link:
        hospital = hospital_link.get_text(' ', strip=True)
    fees = meta.get('priceRange')
    if fees is None:
        fees = parse_fee_from_text(raw_text)

    return {
        'doctor_name': doctor_name,
        'doctor_type': doctor_type,
        'experience_years': experience_years,
        'location': location,
        'hospital': hospital,
        'fees': fees,
        'city': city,
    }


def scrape_city(city, target_count=5, max_pages=3, pause=1.0):
    results = []
    for page in range(1, max_pages + 1):
        url = build_url(city, page=page)
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        cards = soup.select("div.listing-doctor-card[data-qa-id='doctor_card']")
        for card in cards:
            record = parse_card(card, city)
            if record['doctor_name']:
                results.append(record)
            if len(results) >= target_count:
                return results[:target_count]
        time.sleep(pause)
    return results[:target_count]

In [9]:
all_rows = []
for city in CITIES:
    city_rows = scrape_city(city, target_count=50, max_pages=10, pause=1.0)
    all_rows.extend(city_rows)


df = pd.DataFrame(all_rows).drop_duplicates(subset=['city', 'doctor_name', 'hospital'])
df = df[['doctor_name', 'doctor_type', 'experience_years', 'location', 'hospital', 'fees', 'city']]
output_path = 'practo_doctors_500_entries.csv'
df.to_csv(output_path, index=False)
df.head(10)

,doctor_name,doctor_type,experience_years,location,hospital,fees,city
0,Dr. Anil Agadi,General Surgeon,33,"Wilson Garden , Bangalore",SRV Agadi Hospital,600.0,Bangalore
1,Dr. Rajasekhar,General Surgeon,46,"BTM Layout 2nd Stage , Bangalore",Chirag Global Hospital,3000.0,Bangalore
2,Dr. Eashwernath P S,General Physician,42,"HSR Layout , Bangalore",Greenview Medical Center,1500.0,Bangalore
3,Dr. Tulip Chamany,General Surgeon,35,"Arekere , Bangalore",Camry Hospitals,600.0,Bangalore
4,Dr. Shabeer Ahmed,Laparoscopic Surgeon,40,"Sivanchetti Gardens , Bangalore",Q Medical Centre and Hospital,650.0,Bangalore
5,Dr. Arun Kumar,General Surgeon,36,"HSR Layout , Bangalore",Greenview Medical Center,1500.0,Bangalore
6,Dr. Aditya S Chowti,General Physician,18,"Kalyan Nagar , Bangalore",Trilife Hospital,1200.0,Bangalore
7,Dr. Raghavendra R,Pain Management Specialist,26,"Hebbal , Bangalore",Aster CMI Hospital,1200.0,Bangalore
8,Dr. Ravishankar Reddy C R,General Physician,33,"Koramangala 1 Block , Bangalore",Marvel Multispeciality Hospital,1000.0,Bangalore
9,Dr. Ranga Naik,General Physician,49,"Koramangala 1 Block , Bangalore",Marvel Multispeciality Hospital,1000.0,Bangalore


In [10]:
summary = (
    df.groupby('city', dropna=False)
      .size()
      .reset_index(name='doctor_count')
)
summary
df.shape, summary.shape

((484, 7), (10, 2))

In [12]:
# Save the current dataframe without rerunning the scrape
print(f'rows_before_save={len(df)}')
df.to_csv(output_path, index=False)
print(f'saved={output_path}')

rows_before_save=484
saved=practo_doctors_500_entries.csv
